# Phase 2 - Track MLlib Diagnostics in MLflow

This notebook logs the five verified Spark MLlib diagnostic runs to MLflow so the presentation can compare them in the MLflow UI.

The final-project registry step remains separate: after threshold tuning and chronological validation, the selected model must be logged as an artifact, registered, and promoted through staging to production.

In [1]:
import json
import os
from pathlib import Path

import mlflow
import pandas as pd
from IPython.display import display

METRICS_PATH = Path("/workspace/data/local_cache/model_metrics/january_2024_mllib.json")
EXPERIMENT_NAME = "aviation_disruption_original_data_diagnostics"
TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://mlflow:5000")
LOG_RUNS = False  # Set True only when intentionally creating five new MLflow runs.

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)
results = json.loads(METRICS_PATH.read_text())
assert len(results) >= 5, "The rubric requires at least five tracked runs. Rerun the production experiment job."
print("MLflow tracking URI:", TRACKING_URI)
print("Experiment:", EXPERIMENT_NAME)

MLflow tracking URI: http://mlflow:5000
Experiment: aviation_disruption_original_data_diagnostics


## Log Five Verified Runs

Each MLflow run records model identity, dataset scope, and diagnostic metrics produced by Spark MLlib.

In [2]:
if LOG_RUNS:
    for result in results:
        with mlflow.start_run(run_name=result["run_name"]):
            mlflow.log_params({
            "model_run_name": result["run_name"],
            "dataset_year": result["year"],
            "dataset_month": result["month"],
            "label_source": "BTS_REAL_OUTCOME",
            "evaluation_scope": "january_2024_diagnostic_random_split",
            })
            metric_values = {
                key: value for key, value in result.items()
                if key not in {"run_name", "year", "month"} and isinstance(value, (int, float))
            }
            mlflow.log_metrics(metric_values)
            print("Logged:", result["run_name"])
else:
    print("Safe presentation mode: using existing MLflow runs without creating duplicates.")

print("Open the MLflow UI and compare the five runs:", TRACKING_URI)

Safe presentation mode: using existing MLflow runs without creating duplicates.
Open the MLflow UI and compare the five runs: http://mlflow:5000


In [3]:
client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
runs = client.search_runs([experiment.experiment_id], order_by=["metrics.auc DESC"])
comparison_df = pd.DataFrame([
    {
        "run_name": run.data.tags.get("mlflow.runName"),
        "auc": run.data.metrics.get("auc"),
        "accuracy": run.data.metrics.get("accuracy"),
        "positive_recall": run.data.metrics.get("positive_recall"),
        "status": run.info.status,
    }
    for run in runs
])
display(comparison_df)

,run_name,auc,accuracy,positive_recall,status
0,random_forest_trees_40_depth_8,0.700825,0.732766,0.000585,FINISHED
1,logistic_regression_reg_0_01,0.690043,0.750228,0.144041,FINISHED
2,logistic_regression_reg_0_05,0.684267,0.743423,0.082889,FINISHED
3,logistic_regression_reg_0_10,0.679896,0.738136,0.044058,FINISHED
4,random_forest_trees_20_depth_6,0.677954,0.732656,0.000000,FINISHED


## MLflow Compliance Status

- Five diagnostic MLflow runs: covered by this notebook.
- Model artifact logging: pending after model selection.
- Registry registration and staging/production promotion: pending final-model workflow.